In [1]:
import pyeuropepmc
import requests
import pandas as pd
import numpy as np

In [2]:
all_papers = pd.read_csv("sirt6_paper_corpus/SIRT6_openalex_papers_all_entries.csv")

In [3]:
all_papers.head()

,mag,title,publication_year,doi,pmid,authors,journal,abstract,citation_count,open_access,open_access_link,type,is_retracted,concepts,topics,page
0,2.042717e+09,Genomic Instability and Aging-like Phenotype i...,2006,https://doi.org/10.1016/j.cell.2005.11.044,https://pubmed.ncbi.nlm.nih.gov/16439206,"['Raúl Mostoslavsky', 'Katrin F. Chua', 'David...",Cell,NaN,1597,True,http://www.cell.com/article/S0092867406000493/pdf,article,False,"[{'id': 'https://openalex.org/C86803240', 'wik...","[{'id': 'https://openalex.org/T11051', 'displa...",1
1,2.029970e+09,SIRT6 is a histone H3 lysine 9 deacetylase tha...,2008,https://doi.org/10.1038/nature06736,https://pubmed.ncbi.nlm.nih.gov/18337721,"['Eriko Michishita', 'Ronald A. McCord', 'Elis...",Nature,NaN,1086,True,http://doi.org/10.1038/nature06736,article,False,"[{'id': 'https://openalex.org/C2777595374', 'w...","[{'id': 'https://openalex.org/T11051', 'displa...",1
2,2.063084e+09,SIRT6 Links Histone H3 Lysine 9 Deacetylation ...,2009,https://doi.org/10.1016/j.cell.2008.10.052,https://pubmed.ncbi.nlm.nih.gov/19135889,"['Tiara L.A. Kawahara', 'Eriko Michishita', 'A...",Cell,NaN,1058,True,http://www.cell.com/article/S0092867408014463/pdf,article,False,"[{'id': 'https://openalex.org/C2777595374', 'w...","[{'id': 'https://openalex.org/T11051', 'displa...",1
3,1.994540e+09,The sirtuin SIRT6 regulates lifespan in male mice,2012,https://doi.org/10.1038/nature10815,https://pubmed.ncbi.nlm.nih.gov/22367546,"['Yariv Kanfi', 'Shoshana Naiman', 'Gail Amir'...",Nature,NaN,1041,False,NaN,article,False,"[{'id': 'https://openalex.org/C2777595374', 'w...","[{'id': 'https://openalex.org/T11051', 'displa...",1
4,2.109560e+09,The Histone Deacetylase Sirt6 Regulates Glucos...,2010,https://doi.org/10.1016/j.cell.2009.12.041,https://pubmed.ncbi.nlm.nih.gov/20141841,"['Lei Zhong', 'Agustina D’Urso', 'Debra Toiber...",Cell,NaN,978,True,http://www.cell.com/article/S0092867409016274/pdf,article,False,"[{'id': 'https://openalex.org/C86803240', 'wik...","[{'id': 'https://openalex.org/T11051', 'displa...",1


## Detection of corrupted abstracts

In [4]:
import nltk

nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize, word_tokenize
import xml.etree.ElementTree as ET

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/dmitriismirnov/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Clean PMID entries

In [5]:
all_papers['pmid'] = all_papers.pmid.str.replace("https://pubmed.ncbi.nlm.nih.gov/", "")
all_papers['doi'] = all_papers.doi.str.replace("https://doi.org/", "")

Search for missing PMIDs using esearch:

In [6]:
def doi_to_pmid(doi, email=None):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": f"{doi}[DOI]",
        "retmode": "xml"
    }
    if email:
        params["email"] = email

    r = requests.get(url, params=params)
    r.raise_for_status()

    root = ET.fromstring(r.text)
    pmids = [id_elem.text for id_elem in root.findall(".//Id")]
    return pmids[0] if pmids else "N/A"

In [7]:
missing_pmids = all_papers.loc[pd.isna(all_papers.pmid), 'doi'].apply(doi_to_pmid, email = "cosmoskaluga@yandex.ru")

In [8]:
all_papers.loc[pd.isna(all_papers.pmid), 'pmid'] = missing_pmids

In [9]:
all_papers.to_csv("sirt6_paper_corpus/SIRT6_openalex_papers_updated_pmids.csv", index=False)

In [10]:
def abstract_stats(text):
    if not isinstance(text, str):
        return pd.Series({"n_words": 0, "n_sents": 0})

    return pd.Series({
        "n_words": len(word_tokenize(text)),
        "n_sents": len(sent_tokenize(text))
    })

all_papers[["abstracts_n_words", "abstracts_n_sents"]] = all_papers["abstract"].apply(abstract_stats)

In [11]:
def check_truncated(text):
    if not isinstance(text, str) or len(text) < 20:
        return True
    return not text.strip().endswith((".", "!", "?"))

In [12]:
all_papers[['title', 'pmid', 'abstract', 'abstracts_n_words', 'abstracts_n_sents']].head(20)

,title,pmid,abstract,abstracts_n_words,abstracts_n_sents
0,Genomic Instability and Aging-like Phenotype i...,16439206,NaN,0,0
1,SIRT6 is a histone H3 lysine 9 deacetylase tha...,18337721,NaN,0,0
2,SIRT6 Links Histone H3 Lysine 9 Deacetylation ...,19135889,NaN,0,0
3,The sirtuin SIRT6 regulates lifespan in male mice,22367546,NaN,0,0
4,The Histone Deacetylase Sirt6 Regulates Glucos...,20141841,NaN,0,0
5,SIRT6 Promotes DNA Repair Under Stress by Acti...,21680843,A genome stability regulator integrates DNA re...,12,1
6,SIRT6 regulates TNF-α secretion through hydrol...,23552949,NaN,0,0
7,The Histone Deacetylase SIRT6 Is a Tumor Suppr...,23217706,NaN,0,0
8,Activation of the Protein Deacetylase SIRT6 by...,24052263,NaN,0,0
9,Mouse Sir2 Homolog SIRT6 Is a Nuclear ADP-ribo...,15795229,Members of the Sir2 family of NAD-dependent pr...,261,12


In [13]:
SIRT6_TERMS = [
    "sirt6",
    "sirtuin 6",
    "sirtuin-6", 
    "sir2", 
    "sirtuin6",
    "sirt 6", 
    "sirt-6"
]

In [14]:
all_papers["flag_short_abstract"] = ((all_papers["abstracts_n_words"] < 20) | (all_papers["abstracts_n_sents"] < 2))
all_papers["flag_truncated"] = all_papers["abstract"].apply(check_truncated)
#all_papers["flag_no_sirt6"] = ~ all_papers["abstract"].str.lower().str.contains("|".join(SIRT6_TERMS), na=False)

In [15]:
all_papers["flag_bad_abstract"] = (all_papers["flag_short_abstract"] | all_papers["flag_truncated"])

In [16]:
all_papers[['title', 'pmid', 'abstract', 'abstracts_n_words', 'abstracts_n_sents', 'flag_short_abstract', 'flag_truncated', 'flag_bad_abstract']].head(20)

,title,pmid,abstract,abstracts_n_words,abstracts_n_sents,flag_short_abstract,flag_truncated,flag_bad_abstract
0,Genomic Instability and Aging-like Phenotype i...,16439206,NaN,0,0,True,True,True
1,SIRT6 is a histone H3 lysine 9 deacetylase tha...,18337721,NaN,0,0,True,True,True
2,SIRT6 Links Histone H3 Lysine 9 Deacetylation ...,19135889,NaN,0,0,True,True,True
3,The sirtuin SIRT6 regulates lifespan in male mice,22367546,NaN,0,0,True,True,True
4,The Histone Deacetylase Sirt6 Regulates Glucos...,20141841,NaN,0,0,True,True,True
5,SIRT6 Promotes DNA Repair Under Stress by Acti...,21680843,A genome stability regulator integrates DNA re...,12,1,True,False,True
6,SIRT6 regulates TNF-α secretion through hydrol...,23552949,NaN,0,0,True,True,True
7,The Histone Deacetylase SIRT6 Is a Tumor Suppr...,23217706,NaN,0,0,True,True,True
8,Activation of the Protein Deacetylase SIRT6 by...,24052263,NaN,0,0,True,True,True
9,Mouse Sir2 Homolog SIRT6 Is a Nuclear ADP-ribo...,15795229,Members of the Sir2 family of NAD-dependent pr...,261,12,False,False,False


In [17]:
all_papers['flag_bad_abstract'].value_counts()

flag_bad_abstract
False    1131
True      894
Name: count, dtype: int64

In [18]:
all_papers['flag_short_abstract'].value_counts()

flag_short_abstract
False    1214
True      811
Name: count, dtype: int64

In [19]:
pmids_to_search = all_papers.loc[all_papers["flag_short_abstract"], "pmid"].tolist()

In [20]:
pmids_to_search[-20] == "N/A"

True

In [21]:
pd.isna(float(pmids_to_search[14]))

False

In [22]:
def get_abstract_by_pmid(pmid):
    if pd.isna(pmid) or pmid == "" or pmid == "N/A":
        return None
    
    else:
        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        params = {
            "db": "pubmed",
         "id": pmid,
         "retmode": "xml"
        }
        r = requests.get(url, params=params)
        r.raise_for_status()

        return r.text

def extract_abstract(xml_text):
    if xml_text is None:
        return None

    root = ET.fromstring(xml_text)
    abstracts = root.findall(".//AbstractText")

    abstract_texts = []
    for a in abstracts:
        text = "".join(a.itertext())
        if text:
            abstract_texts.append(text.strip())

    return " ".join(abstract_texts)

In [23]:
missing_abstracts = [extract_abstract(get_abstract_by_pmid(pmid)) for pmid in pmids_to_search]

In [24]:
all_papers.loc[all_papers["flag_short_abstract"], "abstract"] = missing_abstracts

In [25]:
new_statistic = all_papers["abstract"].apply(abstract_stats)
new_statistic.loc[new_statistic["n_sents"] > 1].shape

(1853, 2)

In [26]:
new_statistic.loc[new_statistic["n_sents"] > 1].shape[0]/all_papers.shape[0]

0.9150617283950617

In [27]:
all_papers["flag_no_sirt6"] = ~ all_papers["abstract"].str.lower().str.contains("|".join(SIRT6_TERMS), na=False)

In [28]:
all_papers.flag_no_sirt6.value_counts()

flag_no_sirt6
False    1786
True      239
Name: count, dtype: int64

In [29]:
all_papers = all_papers.loc[all_papers.flag_no_sirt6 == False]

In [31]:
all_papers[['doi', 'title', 'pmid', 'abstract', 'abstracts_n_words', 'abstracts_n_sents']].head(20)

,doi,title,pmid,abstract,abstracts_n_words,abstracts_n_sents
0,10.1016/j.cell.2005.11.044,Genomic Instability and Aging-like Phenotype i...,16439206,The Sir2 histone deacetylase functions as a ch...,0,0
1,10.1038/nature06736,SIRT6 is a histone H3 lysine 9 deacetylase tha...,18337721,The Sir2 deacetylase regulates chromatin silen...,0,0
2,10.1016/j.cell.2008.10.052,SIRT6 Links Histone H3 Lysine 9 Deacetylation ...,19135889,Members of the sirtuin (SIRT) family of NAD-de...,0,0
3,10.1038/nature10815,The sirtuin SIRT6 regulates lifespan in male mice,22367546,The significant increase in human lifespan dur...,0,0
4,10.1016/j.cell.2009.12.041,The Histone Deacetylase Sirt6 Regulates Glucos...,20141841,SIRT6 is a member of a highly conserved family...,0,0
5,10.1126/science.1202723,SIRT6 Promotes DNA Repair Under Stress by Acti...,21680843,Sirtuin 6 (SIRT6) is a mammalian homolog of th...,12,1
6,10.1038/nature12038,SIRT6 regulates TNF-α secretion through hydrol...,23552949,The Sir2 family of enzymes or sirtuins are kno...,0,0
7,10.1016/j.cell.2012.10.047,The Histone Deacetylase SIRT6 Is a Tumor Suppr...,23217706,Reprogramming of cellular metabolism is a key ...,0,0
8,10.1074/jbc.c113.511261,Activation of the Protein Deacetylase SIRT6 by...,24052263,Mammalian sirtuins (SIRT1 through SIRT7) are m...,0,0
9,10.1074/jbc.m413296200,Mouse Sir2 Homolog SIRT6 Is a Nuclear ADP-ribo...,15795229,Members of the Sir2 family of NAD-dependent pr...,261,12


## Check truncated titles

In [34]:
all_papers[["titles_n_words", "titles_n_sents"]] = all_papers["title"].apply(abstract_stats)

In [35]:
all_papers[["title", "titles_n_words", "titles_n_sents"]] 

,title,titles_n_words,titles_n_sents
0,Genomic Instability and Aging-like Phenotype i...,11,1
1,SIRT6 is a histone H3 lysine 9 deacetylase tha...,12,1
2,SIRT6 Links Histone H3 Lysine 9 Deacetylation ...,15,1
3,The sirtuin SIRT6 regulates lifespan in male mice,8,1
4,The Histone Deacetylase Sirt6 Regulates Glucos...,9,1
...,...,...,...
2020,Dynamic effects of sleep deprivation on emotio...,21,1
2021,Dynamic regulation of TBK1 lactylation shapes ...,9,1
2022,From ketogenic metabolism to targeted therapeu...,11,1
2023,Inflammation-targeted single-atom nanozymes dr...,20,1


In [36]:
all_papers["flag_short_title"] = (all_papers["titles_n_words"] < 2)

In [47]:
title_pmids_to_find = all_papers.loc[all_papers["flag_short_title"] == True].pmid

In [45]:
def extract_titles(xml_text):
    if xml_text is None:
        return None

    root = ET.fromstring(xml_text)
    abstracts = root.findall(".//ArticleTitle")

    abstract_texts = []
    for a in abstracts:
        text = "".join(a.itertext())
        if text:
            abstract_texts.append(text.strip())

    return " ".join(abstract_texts)

In [50]:
missing_titles = [extract_titles(get_abstract_by_pmid(pmid)) for pmid in title_pmids_to_find]


In [51]:
all_papers.loc[all_papers["flag_short_title"], "title"] = missing_titles

check titles containing stop words

In [64]:
stop_words = ["WITHDRAWN", "Supporting Information", "Figure", "literature review"]

In [65]:
titles_with_stop_words = [
    sentence for sentence in all_papers.title.values 
    if any(stop_word in sentence for stop_word in stop_words)
]

In [66]:
titles_with_stop_words

['Role of sirtuin 6 in early vascular aging in young and middle-aged patients with coronary artery disease (literature review)']

In [67]:
all_papers = all_papers.loc[~all_papers.title.isin(titles_with_stop_words)]

In [69]:
all_papers.to_csv("sirt6_paper_corpus/SIRT6_openalex_papers_recovered_abstracts.csv", index=False)

## Save to json

In [70]:
import json

In [71]:
papers_dict = {}

for idx, row in all_papers.iterrows():
    paper_id = row.get("title", f'paper_{idx}')
    papers_dict[paper_id] = row["abstract"]

with open("sirt6_paper_corpus/sirt6_paper_abstracts.json", "w", encoding="utf-8") as f:
    json.dump(papers_dict, f, indent = 4, ensure_ascii = False)